# 03 Baseline Forecasting

## Project context

This notebook builds transparent one-month-ahead baseline forecasts for Philippines inflation using the cleaned monthly macro indicator table from Phase 2. The goal is not to build an advanced nowcasting system yet; it is to create a clear benchmark layer for later policy interpretation.

## Forecasting objective

- Target: next-month `inflation_rate`.
- Split: chronological train/test split, with the final 20 percent of observations used as test data.
- Models compared: naive last-value benchmark, 3-month moving-average benchmark, and simple linear regression using available lag, rolling, change, and USD/PHP features.
- Output: test predictions, forecast metrics, latest one-month-ahead forecast, and forecast diagnostic figures.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import FIGURES_DIR, OUTPUTS_DIR, PROCESSED_DATA_DIR
from src.forecasting import (
    build_forecast_comparison,
    chronological_train_test_split,
    create_forecast_target,
    create_next_month_forecast,
    evaluate_forecast,
    load_monthly_indicators,
    moving_average_forecast,
    naive_last_value_forecast,
    regression_forecast,
)
from src.visualization import (
    plot_actual_vs_forecast,
    plot_forecast_errors,
    plot_forecast_metrics,
    plot_latest_forecast_context,
)

FORECAST_DIR = OUTPUTS_DIR / "forecasts"
FORECAST_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, path):
    """Save a Plotly figure as PNG, with an honest fallback if static export is unavailable."""
    try:
        fig.write_image(str(path), width=1200, height=700, scale=2)
    except Exception as exc:
        try:
            from PIL import Image, ImageDraw

            image = Image.new("RGB", (1200, 700), "white")
            draw = ImageDraw.Draw(image)
            title = fig.layout.title.text if fig.layout.title.text else path.stem
            draw.text((40, 40), title, fill=(17, 24, 39))
            draw.text((40, 90), "Plotly static export unavailable in this environment.", fill=(75, 85, 99))
            draw.text((40, 125), f"Original export error: {type(exc).__name__}", fill=(75, 85, 99))
            image.save(path)
        except Exception:
            raise exc

## Load monthly indicator table

In [ ]:
input_path = PROCESSED_DATA_DIR / "monthly_macro_indicators.csv"
monthly = load_monthly_indicators(input_path)
monthly.shape, monthly.head(), monthly.tail()

## Define one-month-ahead inflation target

In [ ]:
forecast_df = create_forecast_target(monthly, target_col="inflation_rate", horizon=1)
target_col = "inflation_target_1m"

candidate_features = [
    "inflation_rate_lag_1",
    "inflation_rate_lag_3",
    "inflation_rate_lag_6",
    "inflation_rate_rolling_3",
    "inflation_rate_rolling_6",
    "inflation_rate_change_1",
    "usd_php_lag_1",
    "usd_php_change_1",
]
feature_cols = [
    col for col in candidate_features
    if col in forecast_df.columns and forecast_df[col].notna().sum() > 0
]

modeling_df = forecast_df.dropna(subset=["inflation_rate", target_col, *feature_cols]).copy()
feature_cols, modeling_df.shape, modeling_df[["date", "target_date", "inflation_rate", target_col]].tail()

## Train-test split

The split is chronological. The final 20 percent of usable monthly observations are held out as test data to avoid look-ahead leakage from random sampling.

In [ ]:
train, test = chronological_train_test_split(modeling_df, test_size=0.2)
split_summary = {
    "train_rows": len(train),
    "test_rows": len(test),
    "train_start": train["date"].min(),
    "train_end": train["date"].max(),
    "test_start": test["date"].min(),
    "test_end": test["date"].max(),
}
split_summary

## Naive last-value benchmark

The naive benchmark assumes next-month inflation equals the current observed inflation rate at the forecast origin.

In [ ]:
naive_pred = pd.Series(
    naive_last_value_forecast(train, test, target_col="inflation_rate"),
    index=test.index,
    name="naive_last_value_forecast",
)

## 3-month moving average benchmark

In [ ]:
moving_average_pred = pd.Series(
    moving_average_forecast(train, test, target_col="inflation_rate", window=3),
    index=test.index,
    name="moving_average_3m_forecast",
)

## Simple regression model

The regression uses available lag, rolling, change, and USD/PHP features. The local environment has a known sklearn/SciPy compatibility issue, so the project utilities use NumPy least squares as a transparent fallback when sklearn cannot be imported.

In [ ]:
regression_pred, regression_model = regression_forecast(
    train,
    test,
    feature_cols=feature_cols,
    target_col=target_col,
)
regression_pred = regression_pred.rename("linear_regression_forecast")
regression_model.model_type, regression_model.intercept, dict(zip(feature_cols, regression_model.coefficients))

## Forecast evaluation

In [ ]:
prediction_df = test[["date", "target_date", "inflation_rate", target_col]].copy()
prediction_df = prediction_df.rename(columns={target_col: "actual_inflation_1m_ahead"})
prediction_df["naive_last_value_forecast"] = naive_pred
prediction_df["moving_average_3m_forecast"] = moving_average_pred
prediction_df["linear_regression_forecast"] = regression_pred

forecast_cols = [
    "naive_last_value_forecast",
    "moving_average_3m_forecast",
    "linear_regression_forecast",
]
for col in forecast_cols:
    prediction_df[f"{col}_error"] = prediction_df[col] - prediction_df["actual_inflation_1m_ahead"]

metrics = {
    "naive_last_value": evaluate_forecast(
        prediction_df["actual_inflation_1m_ahead"],
        prediction_df["naive_last_value_forecast"],
        reference=prediction_df["inflation_rate"],
    ),
    "moving_average_3m": evaluate_forecast(
        prediction_df["actual_inflation_1m_ahead"],
        prediction_df["moving_average_3m_forecast"],
        reference=prediction_df["inflation_rate"],
    ),
    "linear_regression": evaluate_forecast(
        prediction_df["actual_inflation_1m_ahead"],
        prediction_df["linear_regression_forecast"],
        reference=prediction_df["inflation_rate"],
    ),
}
metrics_df = build_forecast_comparison(metrics).sort_values("rmse").reset_index(drop=True)
best_model = metrics_df.iloc[0]["model"]
metrics_df

## Actual versus predicted inflation

In [ ]:
actual_vs_forecast_fig = plot_actual_vs_forecast(
    prediction_df,
    date_col="target_date",
    actual_col="actual_inflation_1m_ahead",
    forecast_cols=forecast_cols,
    title="Actual vs Forecast One-Month-Ahead Inflation",
)
actual_vs_forecast_fig.show()
save_figure(actual_vs_forecast_fig, FIGURES_DIR / "inflation_actual_vs_forecast.png")

## Forecast error interpretation

In [ ]:
error_cols = [f"{col}_error" for col in forecast_cols]
error_fig = plot_forecast_errors(
    prediction_df,
    date_col="target_date",
    error_cols=error_cols,
    title="Forecast Errors by Model",
)
error_fig.show()
save_figure(error_fig, FIGURES_DIR / "forecast_error_by_model.png")

metrics_fig = plot_forecast_metrics(metrics_df, metric_col="rmse")
metrics_fig.show()
save_figure(metrics_fig, FIGURES_DIR / "forecast_metrics_comparison.png")

## Latest one-month-ahead forecast

In [ ]:
latest_row = forecast_df.dropna(subset=feature_cols + ["inflation_rate"]).iloc[-1]
latest_forecast_value = create_next_month_forecast(regression_model, latest_row, feature_cols)
latest_forecast_date = pd.to_datetime(latest_row["date"]) + pd.DateOffset(months=1)

latest_forecast = pd.DataFrame(
    [
        {
            "forecast_origin_date": latest_row["date"],
            "forecast_target_date": latest_forecast_date,
            "model": "linear_regression",
            "model_backend": regression_model.model_type,
            "latest_observed_inflation_rate": latest_row["inflation_rate"],
            "forecast_inflation_rate": latest_forecast_value,
            "features_used": ", ".join(feature_cols),
        }
    ]
)
latest_forecast

In [ ]:
context_history = monthly.dropna(subset=["inflation_rate"]).tail(36).copy()
latest_context_fig = plot_latest_forecast_context(
    context_history,
    date_col="date",
    value_col="inflation_rate",
    forecast_date=latest_forecast_date,
    forecast_value=latest_forecast_value,
)
latest_context_fig.show()
save_figure(latest_context_fig, FIGURES_DIR / "latest_forecast_context.png")

## Save Phase 3 outputs

In [ ]:
prediction_output = prediction_df.copy()
for date_col in ["date", "target_date"]:
    prediction_output[date_col] = pd.to_datetime(prediction_output[date_col]).dt.strftime("%Y-%m-%d")
prediction_output.to_csv(FORECAST_DIR / "inflation_forecast_test_predictions.csv", index=False)

metrics_output = metrics_df.copy()
metrics_output.to_csv(FORECAST_DIR / "forecast_metrics.csv", index=False)

latest_output = latest_forecast.copy()
latest_output["forecast_origin_date"] = pd.to_datetime(latest_output["forecast_origin_date"]).dt.strftime("%Y-%m-%d")
latest_output["forecast_target_date"] = pd.to_datetime(latest_output["forecast_target_date"]).dt.strftime("%Y-%m-%d")
latest_output.to_csv(FORECAST_DIR / "latest_inflation_forecast.csv", index=False)

list(FORECAST_DIR.glob("*.csv"))

## Business and policy interpretation

The baselines create a practical reference point for macro monitoring. A naive benchmark tests inflation persistence, the moving-average benchmark smooths short-run volatility, and the regression benchmark tests whether lagged inflation momentum and USD/PHP movements add explanatory signal. These outputs are appropriate for a portfolio research workflow, but they are not a policy model or a production nowcasting system.

## Limitations

- The model uses a simple historical monthly table and does not include policy rates, survey expectations, oil prices, rice prices, or other high-frequency indicators.
- The regression is a linear baseline, not an advanced nowcasting model.
- Evaluation uses a single chronological holdout period, not rolling-origin validation.
- Directional accuracy is only a rough indicator of whether the model captures month-to-month inflation movement.

## Next steps for Phase 4 policy interpretation

- Translate forecast outputs into policy and business implications.
- Summarize what the baseline forecast says about inflation pressure.
- Document how the forecast should and should not be used.
- Prepare dashboard-ready outputs only after interpretation is complete.